# 02 - Stage 1: the pre-assessment band

The customer wants a number before anything is verified. A single rate quoted at
that point is a promise the file cannot keep, so Stage 1 produces an interval,
an eligibility range and a confidence level.

Quantile regression at the 10th and 90th percentiles gives the band; the median
model gives the anchor that Stage 2 is later compared against.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

from aavas.ingestion import loader
df = loader.load_training_table()
print(f"{len(df):,} applications, {df.application_date.min():%b %Y} to {df.application_date.max():%b %Y}")


## What Stage 1 is allowed to see

The feature contract is enforced in code: a Stage-2 field cannot leak into a
quote made before that field exists.

In [ ]:
from aavas.features import stage1_features
from aavas.ingestion.validator import assert_no_leakage

spec = stage1_features.stage1_spec()
assert_no_leakage(spec.columns, "stage1")
print(f"{len(spec.columns)} pre-assessment features:")
print("  " + ", ".join(spec.columns))


## Band quality

Two things matter and they pull against each other: the band has to contain the
final rate often enough to be honest, and be narrow enough to be useful.

In [ ]:
from aavas.models.stage1.predict import Stage1PricingModel
from aavas.models import base

model = Stage1PricingModel.load()
_, test = loader.train_test_split(df)
pred = model.predict_frame(test)
actual = test.sanctioned_rate.to_numpy()

metrics = base.interval_metrics(actual, pred.min_rate.to_numpy(), pred.max_rate.to_numpy())
for key, value in metrics.items():
    print(f"  {key:<22} {value:.2f}")
print()
print(model.card.limitations)


## Coverage by slice

An overall coverage number can hide a segment the band systematically misses.

In [ ]:
from aavas.models.stage1.evaluate import evaluate
evaluate(df)


## One customer's quote

In [ ]:
quote = model.predict(test.head(1))
print(f"Indicative range  {quote['band']['min_rate']:.2f}% - {quote['band']['max_rate']:.2f}%")
print(f"Confidence        {quote['confidence']:.2f}")
print(f"Eligible amount   Rs {quote['eligible_amount']['min']:,.0f} - Rs {quote['eligible_amount']['max']:,.0f}")
print(f"Actual outcome    {test.head(1).sanctioned_rate.iloc[0]:.2f}%")


## Confidence should track information

A thin file should get a lower confidence score, not a confidently wrong band.

In [ ]:
known = test[test.bureau_score > 0].head(500)
thin = known.assign(bureau_score=-1, new_to_credit=True, competitor_rate=np.nan)
print(f"Established credit file: {model.predict_frame(known).confidence.mean():.3f}")
print(f"New to credit, no quote: {model.predict_frame(thin).confidence.mean():.3f}")
